# Llama Vision Table Extraction

Self-contained notebook for extracting structured table data from images using Llama-3.2-Vision-Instruct.

In [ ]:
# Configuration
MODEL_PATH = "/path/to/Llama-3.2-11B-Vision-Instruct"  # UPDATE THIS PATH
IMAGE_PATH = "path/to/your/image.jpg"  # UPDATE THIS PATH

# GPU settings
LOAD_IN_8BIT = True  # Enable 8-bit quantization for memory efficiency
DEVICE_MAP = "auto"  # Automatic device mapping

In [ ]:
# Install required packages if needed
# !pip install transformers torch torchvision pillow accelerate bitsandbytes

In [ ]:
import json
import re
from typing import Any, Dict, List

import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, MllamaForConditionalGeneration

In [ ]:
# Initialize model and processor
print("Loading Llama Vision model...")

model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE_MAP,
    load_in_8bit=LOAD_IN_8BIT if LOAD_IN_8BIT else None,
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("✅ Model loaded successfully")

In [ ]:
# Table extraction prompt templates
TABLE_EXTRACTION_PROMPTS = {
    "structured_json": """Analyze the table in this image and extract all data in a structured JSON format.

Return the table data as a JSON object with the following structure:
{
  "columns": [list of column headers],
  "rows": [
    {"column1": "value1", "column2": "value2", ...},
    {"column1": "value1", "column2": "value2", ...}
  ]
}

Extract ALL visible data from the table. Be precise with numbers and preserve formatting.""",
    
    "csv_format": """Extract the table from this image and format it as CSV.

Rules:
1. First line should be column headers separated by commas
2. Each subsequent line is a data row
3. Use quotes around values containing commas
4. Extract ALL rows and columns visible in the table
5. Preserve number formatting and units

Return ONLY the CSV data, no additional text.""",
    
    "markdown_table": """Convert the table in this image to Markdown table format.

Format:
| Column1 | Column2 | Column3 |
|---------|---------|---------|  
| Value1  | Value2  | Value3  |

Extract ALL data from the table exactly as shown.""",
    
    "detailed_extraction": """Carefully analyze the table in this image and extract all information.

For each table found:
1. Table title/caption (if present)
2. Column headers (in order)
3. All data rows (preserve original order)
4. Any footer/summary rows
5. Notes or annotations

Format your response as:
TABLE:
Title: [title if present]
Columns: [list all column headers]
Row 1: [all values in first row]
Row 2: [all values in second row]
[continue for all rows]
Notes: [any additional notes]

Be extremely precise with numbers, dates, and formatting.""",
    
    "invoice_table": """Extract the line items table from this invoice/receipt image.

For each line item, extract:
- Description/Item name
- Quantity
- Unit price
- Total/Amount
- Any additional columns (tax, discount, etc.)

Format as JSON:
{
  "line_items": [
    {
      "description": "...",
      "quantity": "...",
      "unit_price": "...",
      "total": "..."
    }
  ],
  "subtotal": "...",
  "tax": "...",
  "total": "..."
}"""
}

In [ ]:
def extract_table(image_path: str, prompt_type: str = "structured_json", custom_prompt: str = None) -> Dict[str, Any]:
    """
    Extract table data from an image using Llama Vision.
    
    Args:
        image_path: Path to the image file
        prompt_type: Type of extraction prompt to use
        custom_prompt: Optional custom prompt to override defaults
    
    Returns:
        Dictionary containing extracted data and metadata
    """
    # Load image
    image = Image.open(image_path)
    
    # Select prompt
    prompt = custom_prompt if custom_prompt else TABLE_EXTRACTION_PROMPTS.get(prompt_type, TABLE_EXTRACTION_PROMPTS["structured_json"])
    
    # Prepare inputs
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Apply chat template
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    
    # Process inputs
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            temperature=0.1,
            top_p=0.95
        )
    
    # Decode response
    response = processor.decode(output[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    
    return {
        "raw_response": response,
        "prompt_type": prompt_type,
        "image_path": image_path
    }

In [ ]:
def parse_json_response(response: str) -> Dict:
    """
    Parse JSON from model response.
    """
    try:
        # Try to find JSON in the response
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass
    return None

def parse_csv_response(response: str) -> pd.DataFrame:
    """
    Parse CSV format response into DataFrame.
    """
    try:
        lines = response.strip().split('\n')
        # Filter out non-CSV lines
        csv_lines = [line for line in lines if ',' in line]
        if csv_lines:
            # Create DataFrame from CSV text
            from io import StringIO
            csv_text = '\n'.join(csv_lines)
            return pd.read_csv(StringIO(csv_text))
    except:
        pass
    return None

def parse_markdown_table(response: str) -> pd.DataFrame:
    """
    Parse Markdown table format into DataFrame.
    """
    try:
        lines = response.strip().split('\n')
        table_lines = [line for line in lines if '|' in line]
        
        if len(table_lines) >= 3:  # Header, separator, at least one data row
            # Parse header
            header = [cell.strip() for cell in table_lines[0].split('|') if cell.strip()]
            
            # Parse data rows (skip separator line)
            data = []
            for line in table_lines[2:]:
                row = [cell.strip() for cell in line.split('|') if cell.strip()]
                if len(row) == len(header):
                    data.append(row)
            
            if data:
                return pd.DataFrame(data, columns=header)
    except:
        pass
    return None

In [ ]:
# Extract table using structured JSON format
print("Extracting table from image...")
result = extract_table(IMAGE_PATH, prompt_type="structured_json")

print("\n📊 Raw Response:")
print("=" * 50)
print(result["raw_response"])
print("=" * 50)

In [ ]:
# Parse and display structured data
parsed_data = parse_json_response(result["raw_response"])

if parsed_data:
    print("✅ Successfully parsed JSON data")
    
    # Convert to DataFrame for better visualization
    if "rows" in parsed_data:
        df = pd.DataFrame(parsed_data["rows"])
        print("\n📊 Extracted Table:")
        display(df)
        
        # Save to CSV
        output_file = "extracted_table.csv"
        df.to_csv(output_file, index=False)
        print(f"\n💾 Saved to {output_file}")
    
    # Display full parsed structure
    print("\n📋 Full Structure:")
    print(json.dumps(parsed_data, indent=2))
else:
    print("⚠️ Could not parse JSON. Try CSV or markdown format.")

In [ ]:
# Try CSV format extraction
print("Trying CSV format extraction...")
csv_result = extract_table(IMAGE_PATH, prompt_type="csv_format")

print("\n📊 CSV Response:")
print("=" * 50)
print(csv_result["raw_response"])
print("=" * 50)

# Parse CSV
csv_df = parse_csv_response(csv_result["raw_response"])
if csv_df is not None:
    print("\n✅ Parsed CSV Table:")
    display(csv_df)

In [ ]:
# Try Markdown format extraction  
print("Trying Markdown format extraction...")
md_result = extract_table(IMAGE_PATH, prompt_type="markdown_table")

print("\n📊 Markdown Response:")
print("=" * 50)
print(md_result["raw_response"])
print("=" * 50)

# Parse Markdown table
md_df = parse_markdown_table(md_result["raw_response"])
if md_df is not None:
    print("\n✅ Parsed Markdown Table:")
    display(md_df)

In [ ]:
# Custom prompt example for specific use cases
custom_prompt = """
Extract all numerical data from the table in this image.
Focus on:
1. All column headers
2. All numerical values (preserve decimal places)
3. Any units or currency symbols
4. Row labels/identifiers

Format as a clean JSON object with arrays for each column.
"""

custom_result = extract_table(IMAGE_PATH, custom_prompt=custom_prompt)
print("\n📊 Custom Extraction Result:")
print(custom_result["raw_response"])

In [ ]:
# Batch processing function for multiple images
def batch_extract_tables(image_paths: List[str], prompt_type: str = "structured_json") -> List[Dict]:
    """
    Extract tables from multiple images.
    """
    results = []
    
    for i, image_path in enumerate(image_paths, 1):
        print(f"\nProcessing image {i}/{len(image_paths)}: {image_path}")
        try:
            result = extract_table(image_path, prompt_type)
            
            # Try to parse the result
            parsed = parse_json_response(result["raw_response"])
            if parsed:
                result["parsed_data"] = parsed
                print(f"  ✅ Successfully extracted table with {len(parsed.get('rows', []))} rows")
            else:
                print("  ⚠️ Extracted but could not parse structured data")
                
            results.append(result)
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            results.append({"error": str(e), "image_path": image_path})
    
    return results

# Example batch processing (uncomment and update paths)
# image_list = ["table1.jpg", "table2.jpg", "table3.jpg"]
# batch_results = batch_extract_tables(image_list)
# print(f"\n✅ Processed {len(batch_results)} images")

In [ ]:
# Advanced: Invoice/Receipt specific extraction
print("\n📄 Invoice Table Extraction:")
invoice_result = extract_table(IMAGE_PATH, prompt_type="invoice_table")

invoice_data = parse_json_response(invoice_result["raw_response"])
if invoice_data:
    print("✅ Extracted Invoice Data:")
    
    # Display line items
    if "line_items" in invoice_data:
        items_df = pd.DataFrame(invoice_data["line_items"])
        print("\n📊 Line Items:")
        display(items_df)
        
        # Display totals
        print("\n💰 Totals:")
        for key in ["subtotal", "tax", "total"]:
            if key in invoice_data:
                print(f"  {key.capitalize()}: {invoice_data[key]}")
else:
    print("Raw invoice response:")
    print(invoice_result["raw_response"])

In [ ]:
# Utility: Clean up GPU memory
def cleanup_gpu():
    """
    Clean up GPU memory after processing.
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 GPU memory cleaned")
        print(f"   Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"   Reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

# cleanup_gpu()

## Tips for Better Table Extraction

1. **Image Quality**: Higher resolution images yield better results
2. **Table Clarity**: Clear borders and headers improve extraction accuracy
3. **Prompt Engineering**: Customize prompts for specific table types
4. **Format Selection**: 
   - Use JSON for structured data processing
   - Use CSV for spreadsheet compatibility
   - Use Markdown for documentation
5. **Post-Processing**: Always validate extracted numerical data

### Common Table Types
- **Financial**: Invoices, receipts, statements
- **Data Tables**: Reports, analytics, summaries  
- **Inventory**: Product lists, catalogs
- **Comparison**: Feature matrices, pricing tables

### Troubleshooting
- **Incomplete extraction**: Try increasing `max_new_tokens`
- **Format issues**: Experiment with different prompt types
- **Memory errors**: Enable 8-bit quantization or reduce batch size